In [1]:
!pip install xgboost
!pip install lightgbm

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   ---------------------------------------- 0.3/101.7 MB ? eta -:--:--
   ---------------------------------------- 1.0/101.7 MB 3.8 MB/s eta 0:00:27
   - -------------------------------------- 2.9/101.7 MB 6.1 MB/s eta 0:00:17
   - -------------------------------------- 4.5/101.7 MB 6.8 MB/s eta 0:00:15
   - -------------------------------------- 4.7/101.7 MB 6.9 MB/s eta 0:00:14
   -- ------------------------------------- 6.0/101.7 MB 5.5 MB/s eta 0:00:18
   -- ------------------------------------- 7.1/101.7 MB 5.4 MB/s eta 0:00:18
   --- ------------------------------------ 8.4/101.7 MB 5.6 MB/s eta 0:00:17
   --- ------------------------------------ 10.0/101.7 MB 5.8 MB/s eta 0:00:16
   ---- ----------------------------------- 11.5/101.7 MB 6.0 MB/s eta 0:00:16
   ----- ---------------------------------- 13.1/101.7 MB 6.1 MB/s eta 0:00:15
   -


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.5 MB ? eta -:--:--
   -------------- ------------------------- 0.5/1.5 MB 3.2 MB/s eta 0:00:01
   ---------------------------------------- 1.5/1.5 MB 5.0 MB/s  0:00:00



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [40]:
# CELL 1: Imports and Setup
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter
from numpy.polynomial import Polynomial
from sklearn.preprocessing import normalize
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import sys

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [41]:
# CELL 2: Data Loading & Transposing
# Assuming the CSV has Wavelengths in the first column and Samples as the other columns
# Row 0 (header) contains sample names. Column 0 contains wavelengths.
def load_and_transpose(filepath):
    print(f"Loading data from {filepath}...")
    df = pd.read_csv(filepath, index_col=0) # Wavelengths become the index
    
    # Transpose so that samples are rows (N) and wavelengths are columns (Features)
    X_df = df.T
    
    # Extract numerical matrices
    X_raw = X_df.values.astype(float)
    wavelengths = X_df.columns.values.astype(float)
    sample_names = X_df.index.values
    
    print(f"Shape of X: {X_raw.shape} ({X_raw.shape[0]} samples, {X_raw.shape[1]} features)")
    print(f"Wavelength range: {wavelengths.min():.5f} nm to {wavelengths.max():.5f} nm")
    
    return X_raw, wavelengths, sample_names

# Replace with your actual filename
csv_filename = r"D:\Ryan\UM\Academic\Y4S1\KIE4002 FINAL YEAR PROJECT\Irish_Soil_DataSet\Irish_Soil_DataSet2.csv"
# UNCOMMENT TO RUN:

X_raw, wavelengths, sample_names = load_and_transpose(csv_filename)



Loading data from D:\Ryan\UM\Academic\Y4S1\KIE4002 FINAL YEAR PROJECT\Irish_Soil_DataSet\Irish_Soil_DataSet2.csv...
Shape of X: (320, 12981) (320 samples, 12981 features)
Wavelength range: 197.00101 nm to 973.58912 nm


In [42]:
# CELL 3: Auto-Labelling using Weighted NIST Peak Areas
def generate_weighted_auto_labels(X, w_array, target_nutrient="N"):
    """
    Calculates a weighted sum of peak areas based on NIST_LIBS_NPK_Emission_Peaks.md
    priorities, then applies K-Means to generate Low/Med/High labels.
    """
    # Define peaks, tolerance (window), and weights based on the MD file
    peaks_config = {
        "N": [
            {"centre": 821.634, "tolerance": 0.5, "weight": 1.0}, # Primary (Best R2)
            {"centre": 868.340, "tolerance": 0.5, "weight": 1.0}, # Primary (Doublet)
            {"centre": 868.028, "tolerance": 0.5, "weight": 1.0}, # Primary (Doublet)
            {"centre": 496.398, "tolerance": 0.5, "weight": 0.5}, # Secondary
            {"centre": 746.831, "tolerance": 0.5, "weight": 0.2}, # Low priority (Old baseline)
        ],
        "P": [
            {"centre": 213.618, "tolerance": 0.3, "weight": 1.0}, # Primary
            {"centre": 213.547, "tolerance": 0.3, "weight": 1.0}, # Primary
            {"centre": 253.561, "tolerance": 0.3, "weight": 0.5}, # Secondary
            {"centre": 214.915, "tolerance": 0.3, "weight": 0.5}, # Secondary
        ],
        "K": [
            {"centre": 766.490, "tolerance": 0.5, "weight": 1.0}, # Primary
            {"centre": 769.896, "tolerance": 0.5, "weight": 1.0}, # Primary
            {"centre": 404.414, "tolerance": 0.5, "weight": 0.5}, # Secondary
            {"centre": 404.721, "tolerance": 0.5, "weight": 0.5}, # Secondary
        ]
    }
    
    config = peaks_config[target_nutrient]
    aggregated_areas = np.zeros(X.shape[0])
    
    print(f"Calculating weighted peak areas for {target_nutrient}:")
    for peak in config:
        center = peak["centre"]
        tol = peak["tolerance"]
        weight = peak["weight"]
        
        # Find exact closest wavelength for logging
        idx = np.argmin(np.abs(w_array - center))
        actual_wl = w_array[idx]
        
        # Extract area within window (boolean mask)
        mask = (w_array >= center - tol) & (w_array <= center + tol)
        area = np.sum(X[:, mask], axis=1)
        
        # Add to total weighted area
        aggregated_areas += (area * weight)
        
        print(f"  - Peak {center:.3f} nm (Found: {actual_wl:.4f}): Weight={weight}, Window=±{tol}nm")
        
    # Reshape for scikit-learn
    aggregated_areas = aggregated_areas.reshape(-1, 1)
    
    # Apply K-Means clustering (k=3 for Low, Medium, High)
    kmeans = KMeans(n_clusters=3, random_state=42, n_init=10).fit(aggregated_areas)
    
    # Sort cluster centers to reliably map them to 0 (Low), 1 (Medium), 2 (High)
    sorted_centers = np.argsort(kmeans.cluster_centers_.flatten())
    label_mapping = {old_label: new_label for new_label, old_label in enumerate(sorted_centers)}
    y = np.vectorize(label_mapping.get)(kmeans.labels_)
    
    print(f"\nFinal Label Distribution: Low(0): {(y==0).sum()}, Med(1): {(y==1).sum()}, High(2): {(y==2).sum()}")
    return y

# Generate labels for Nitrogen using the weighted approach
y_target = generate_weighted_auto_labels(X_raw, wavelengths, target_nutrient="N")

Calculating weighted peak areas for N:
  - Peak 821.634 nm (Found: 821.5925): Weight=1.0, Window=±0.5nm
  - Peak 868.340 nm (Found: 868.3798): Weight=1.0, Window=±0.5nm
  - Peak 868.028 nm (Found: 868.0217): Weight=1.0, Window=±0.5nm
  - Peak 496.398 nm (Found: 496.4004): Weight=0.5, Window=±0.5nm
  - Peak 746.831 nm (Found: 746.8356): Weight=0.2, Window=±0.5nm

Final Label Distribution: Low(0): 112, Med(1): 127, High(2): 81


In [43]:
# CELL 4: Phase 1 Pre-Processing (TinyML Constrained)
def preprocess_spectra(X, w_array):
    print("Step 1: Baseline Correction (5th-order Polynomial)...")
    X_base = np.zeros_like(X)
    for i in range(X.shape[0]):
        # Fit 5th order polynomial and subtract it
        p = Polynomial.fit(w_array, X[i], deg=5)
        corrected = X[i] - p(w_array)
        corrected = np.maximum(corrected, 0)
        # corrected is 1D per-sample → use axis=0 (or omit axis)
        smoothed = savgol_filter(corrected, window_length=7, polyorder=3)
        X_base[i] = np.maximum(smoothed, 0)
        
    print("Step 3: Normalization (L2-Norm)...")
    X_norm = normalize(X_base, norm='l2', axis=1)
    
    return X_norm

# Apply preprocessing
X_preprocessed = preprocess_spectra(X_raw, wavelengths)

# Split data into train/test
X_train_full, X_test_full, y_train, y_test = train_test_split(
    X_preprocessed, y_target, test_size=0.3, random_state=42, stratify=y_target
)
print(f"Training set: {X_train_full.shape[0]} samples. Testing set: {X_test_full.shape[0]} samples.")

Step 1: Baseline Correction (5th-order Polynomial)...
Step 3: Normalization (L2-Norm)...
Training set: 224 samples. Testing set: 96 samples.


In [44]:
# CELL 5: Phase 2 Feature Selection (RF Gini Importance)
print("Running Feature Selection (Random Forest Gini)...")
rf_selector = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_selector.fit(X_train_full, y_train)

# Get top 87 features to meet MCU memory constraints
target_feature_count = 87
importances = rf_selector.feature_importances_
top_87_indices = np.argsort(importances)[::-1][:target_feature_count]

print(f"Top 5 selected wavelengths: {wavelengths[top_87_indices][:5]}")

# Slice the datasets to only contain the 87 selected features
X_train = X_train_full[:, top_87_indices]
X_test = X_test_full[:, top_87_indices]
print(f"Reduced Dataset Shape: {X_train.shape}")

Running Feature Selection (Random Forest Gini)...
Top 5 selected wavelengths: [868.64818 926.62084 344.38341 770.28633 371.08654]
Reduced Dataset Shape: (224, 87)


In [45]:
# CELL 6: Model 1 - Random Forest (TinyML Profile)
print("\n--- Training Model 1: Random Forest ---")
# Max depth constrained to 8 to keep model size under ~100 KB
rf_clf = RandomForestClassifier(n_estimators=100, max_depth=8, min_samples_split=5, random_state=42)
rf_clf.fit(X_train, y_train)

rf_preds = rf_clf.predict(X_test)
rf_acc = accuracy_score(y_test, rf_preds)
print(f"Random Forest Test Accuracy: {rf_acc:.4f}")
print("Classification Report:\n", classification_report(y_test, rf_preds))

# Memory estimation (rough approximation for MCU)
rf_memory_kb = (sys.getsizeof(rf_clf) + (100 * 8 * 2**8 * 4)) / 1024 # Approx node struct size
print(f"~ Estimated MCU Flash Size: {rf_memory_kb:.1f} KB")


--- Training Model 1: Random Forest ---
Random Forest Test Accuracy: 0.8750
Classification Report:
               precision    recall  f1-score   support

           0       0.97      0.88      0.92        34
           1       0.82      0.87      0.85        38
           2       0.84      0.88      0.86        24

    accuracy                           0.88        96
   macro avg       0.88      0.88      0.88        96
weighted avg       0.88      0.88      0.88        96

~ Estimated MCU Flash Size: 800.0 KB


In [46]:
# CELL 7: Model 2 - XGBoost
print("\n--- Training Model 2: XGBoost ---")
xgb_clf = XGBClassifier(
    n_estimators=100, 
    max_depth=6, 
    learning_rate=0.1, 
    objective='multi:softprob', 
    random_state=42
)
xgb_clf.fit(X_train, y_train)

xgb_preds = xgb_clf.predict(X_test)
xgb_acc = accuracy_score(y_test, xgb_preds)
print(f"XGBoost Test Accuracy: {xgb_acc:.4f}")
print("Classification Report:\n", classification_report(y_test, xgb_preds))


--- Training Model 2: XGBoost ---
XGBoost Test Accuracy: 0.8750
Classification Report:
               precision    recall  f1-score   support

           0       0.91      0.94      0.93        34
           1       0.86      0.82      0.84        38
           2       0.84      0.88      0.86        24

    accuracy                           0.88        96
   macro avg       0.87      0.88      0.87        96
weighted avg       0.87      0.88      0.87        96



In [47]:
# CELL 8: Model 3 - LightGBM
print("\n--- Training Model 3: LightGBM ---")
lgb_clf = LGBMClassifier(
    n_estimators=100, 
    max_depth=6, 
    learning_rate=0.1, 
    random_state=42,
    verbose=-1
)
lgb_clf.fit(X_train, y_train)

lgb_preds = lgb_clf.predict(X_test)
lgb_acc = accuracy_score(y_test, lgb_preds)
print(f"LightGBM Test Accuracy: {lgb_acc:.4f}")
print("Classification Report:\n", classification_report(y_test, lgb_preds))


--- Training Model 3: LightGBM ---
LightGBM Test Accuracy: 0.8958
Classification Report:
               precision    recall  f1-score   support

           0       0.94      0.94      0.94        34
           1       0.87      0.87      0.87        38
           2       0.88      0.88      0.88        24

    accuracy                           0.90        96
   macro avg       0.89      0.89      0.89        96
weighted avg       0.90      0.90      0.90        96



C:\Users\ryanc\AppData\Roaming\Python\Python313\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [48]:
# CELL 9: Final Benchmark Summary
print("\n=== FINAL BENCHMARK SUMMARY ===")
print(f"Baseline Poly 5th -> SG(7,3) -> L2 Norm -> RF Gini ({target_feature_count} features)")
print("-" * 40)
print(f"Random Forest (MCU-Target) Acc : {rf_acc * 100:.2f}%")
print(f"XGBoost Accuracy                 : {xgb_acc * 100:.2f}%")
print(f"LightGBM Accuracy                : {lgb_acc * 100:.2f}%")
print("-" * 40)
print("Ready for phase 2 emlearn C-header conversion.")


=== FINAL BENCHMARK SUMMARY ===
Baseline Poly 5th -> SG(7,3) -> L2 Norm -> RF Gini (87 features)
----------------------------------------
Random Forest (MCU-Target) Acc : 87.50%
XGBoost Accuracy                 : 87.50%
LightGBM Accuracy                : 89.58%
----------------------------------------
Ready for phase 2 emlearn C-header conversion.
